# 📊 LONQ Ticker Analyzer - Colab

Análisis completo de cualquier ticker con métricas detalladas.

**Solo necesitas ingresar el ticker que quieras analizar.**

In [ ]:
# 1️⃣ SETUP - Ejecuta esta celda primero
import os
import sys

# Clonar repositorio
if not os.path.exists('/content/a_tecn'):
    print('📥 Clonando repositorio...')
    !git clone https://github.com/pfreezv/a_tecn.git 2>&1 | grep -E "(Cloning|done)"
else:
    print('✓ Repositorio ya existe')

# Cambiar al directorio
os.chdir('/content/a_tecn')
sys.path.insert(0, '/content/a_tecn')

# Instalar dependencias
print('📦 Instalando dependencias...')
!pip install -q -r requirements.txt
print('✓ Dependencias instaladas')

In [ ]:
# 2️⃣ IMPORTAR MÓDULOS
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.ticker_analyzer import TickerAnalyzer

# Configurar visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✓ Módulos importados')

In [ ]:
# 3️⃣ INGRESA EL TICKER QUE QUIERES ANALIZAR
ticker = input('Ingresa ticker (ej: AAPL, TSLA, SPY, BTC): ').upper().strip()

if not ticker:
    ticker = 'AAPL'  # Default
    print(f'⚠ Usando default: {ticker}')

print(f'\n📊 Analizando: {ticker}')
print('='*60)

In [ ]:
# 4️⃣ CARGAR DATOS
analyzer = TickerAnalyzer(ticker=ticker)

# Intentar cargar datos
if not analyzer.load_data():
    print(f'\n⚠ No hay datos locales para {ticker}')
    print('\nOpciones:')
    print('  1. Descargar con yfinance (ejecuta la celda siguiente)')
    print('  2. Usar otro ticker del repositorio: AAPL, TSLA, SPY, KO, BTC')
    DATA_AVAILABLE = False
else:
    print(f'✓ Datos cargados')
    DATA_AVAILABLE = True
    
    # Cargar VIX
    vix_loaded = analyzer.load_vix()
    if vix_loaded:
        print('✓ VIX cargado')
    else:
        print('⚠ VIX no disponible (análisis profundo sin VIX)')

In [ ]:
# 5️⃣ DESCARGAR DATOS CON YFINANCE (si no están disponibles)
if not DATA_AVAILABLE:
    print(f'📥 Descargando datos para {ticker}...')
    
    import yfinance as yf
    
    try:
        # Descargar 5 años de datos
        df = yf.download(ticker, period='5y', progress=False)
        
        if df.empty:
            print(f'✗ {ticker} no es un ticker válido')
        else:
            # Guardar localmente
            df.to_csv(f'{ticker.lower()}_data.csv')
            print(f'✓ {len(df)} filas descargadas y guardadas')
            
            # Reintentar cargar
            analyzer = TickerAnalyzer(ticker=ticker)
            if analyzer.load_data():
                print('✓ Datos cargados exitosamente')
                analyzer.load_vix()
                DATA_AVAILABLE = True
    except Exception as e:
        print(f'✗ Error descargando: {e}')

In [ ]:
# 6️⃣ ANÁLISIS RÁPIDO
if DATA_AVAILABLE:
    print('\n⚡ ANÁLISIS RÁPIDO (~2 segundos)\n')
    
    if analyzer.analyze(deep=False):
        analyzer.print_report()
    else:
        print('✗ Error en análisis rápido')
else:
    print('⚠ Carga datos primero con la celda anterior')

In [ ]:
# 7️⃣ ANÁLISIS PROFUNDO (60 segundos)
if DATA_AVAILABLE:
    print('\n🔬 ANÁLISIS PROFUNDO (~60 segundos)\n')
    print('⏳ En progreso...')
    
    if analyzer.analyze(deep=True, threshold=2.0, horizon=10):
        print('\n' + '='*60)
        analyzer.print_report()
        print('='*60)
    else:
        print('✗ Error en análisis profundo')
        print('  Posibles causas:')
        print('  - Datos insuficientes (<500 días)')
        print('  - Error en ensemble o trigger')
else:
    print('⚠ Carga datos primero')

In [ ]:
# 8️⃣ VISUALIZACIONES
if DATA_AVAILABLE and analyzer.fast_result:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{ticker} - Análisis Detallado', fontsize=16, fontweight='bold')
    
    # 1. Desglose de puntos
    ax = axes[0, 0]
    score_breakdown = analyzer.fast_result.score_breakdown
    colors = plt.cm.Set3(np.linspace(0, 1, len(score_breakdown)))
    ax.barh(list(score_breakdown.keys()), list(score_breakdown.values()), color=colors)
    ax.set_xlabel('Puntos')
    ax.set_title('Desglose de Puntuación')
    ax.grid(axis='x', alpha=0.3)
    
    # 2. Métricas clave
    ax = axes[0, 1]
    metrics = {
        'Volatilidad': f"{analyzer.fast_result.vol_annual:.1%}",
        'Autocorr': f"{analyzer.fast_result.autocorr_5d:.3f}",
        'Tendencia R²': f"{analyzer.fast_result.trend_strength:.3f}",
        'Score': f"{analyzer.fast_result.score}/100"
    }
    ax.axis('off')
    y_pos = 0.9
    for key, val in metrics.items():
        ax.text(0.1, y_pos, f'{key}:', fontweight='bold', fontsize=11)
        ax.text(0.6, y_pos, str(val), fontsize=11)
        y_pos -= 0.2
    ax.set_title('Métricas Rápidas', loc='left', fontweight='bold')
    
    # 3. Distribución de precios
    ax = axes[1, 0]
    prices = analyzer.prices
    ax.plot(prices.index, prices.values, linewidth=1.5, color='#2E86AB')
    ax.fill_between(prices.index, prices.values, alpha=0.3, color='#2E86AB')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Precio')
    ax.set_title('Histórico de Precios')
    ax.grid(alpha=0.3)
    
    # 4. Retornos
    ax = axes[1, 1]
    returns = np.log(prices / prices.shift(1)).dropna()
    ax.hist(returns, bins=50, color='#A23B72', alpha=0.7, edgecolor='black')
    ax.axvline(returns.mean(), color='red', linestyle='--', label=f'Media: {returns.mean():.4f}')
    ax.set_xlabel('Retorno Diario')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribución de Retornos')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Sin datos para visualizar')

In [ ]:
# 9️⃣ EXPORTAR RESULTADOS
if DATA_AVAILABLE and analyzer.fast_result:
    import json
    from google.colab import files
    
    # Exportar como JSON
    resultados = analyzer.to_dict()
    
    with open(f'{ticker}_analysis.json', 'w') as f:
        json.dump(resultados, f, indent=2)
    
    # Exportar como CSV
    df = analyzer.get_metrics_dataframe()
    df.to_csv(f'{ticker}_metrics.csv', index=False)
    
    print(f'✓ Archivos guardados:')
    print(f'  - {ticker}_analysis.json')
    print(f'  - {ticker}_metrics.csv')
    print(f'\n📥 Descargando...')
    
    files.download(f'{ticker}_analysis.json')
    files.download(f'{ticker}_metrics.csv')
else:
    print('⚠ Sin datos para exportar')

In [ ]:
# 🔟 COMPARAR MÚLTIPLES TICKERS (OPCIONAL)
tickers_a_comparar = ['AAPL', 'TSLA', 'SPY', 'KO', 'BTC']  # Modificar lista

print('📊 Comparando tickers...')
resultados = []

for t in tickers_a_comparar:
    analyzer_tmp = TickerAnalyzer(ticker=t)
    if analyzer_tmp.load_data():
        if analyzer_tmp.analyze(deep=False):
            resultados.append(analyzer_tmp.get_metrics_dataframe())
            print(f'  ✓ {t}')
        else:
            print(f'  ✗ {t} (análisis falló)')
    else:
        print(f'  ✗ {t} (sin datos)')

if resultados:
    df_comp = pd.concat(resultados, ignore_index=True)
    print(f'\n🏆 RANKING')
    df_sorted = df_comp.sort_values('Score', ascending=False)
    print(df_sorted.to_string(index=False))
else:
    print('⚠ No hay resultados para comparar')

## 📋 Resumen

Has completado un análisis completo del ticker usando LONQ Ticker Analyzer.

### ¿Qué significa la puntuación?

- **≥70**: Excelente candidato para la estrategia
- **50-69**: Buen candidato
- **30-49**: Candidato marginal
- **<30**: No recomendado

### Próximos pasos

1. Cambiar el ticker en la celda 3 para analizar otros
2. Ejecutar análisis profundo en la celda 7 (recomendado)
3. Comparar múltiples tickers en la celda 10
4. Descargar resultados en la celda 9

---

**Más información**: Consulta `COLAB_GUIDE.md` y `README_ANALYZER.md` en el repositorio.